# 🌦️ Rainfall Prediction System (Dual-Engine AI)
### Project 11: Simultaneous Classification & Regression Web Application
- **Dataset**: Official Seattle Weather Dataset (`seattle-weather.csv`)
- **Features**: `temp_max`, `temp_min`, `wind`, `year`, `month`, `day`
- **Algorithms**:
  - **Classification**: Random Forest, XGBoost, Logistic Regression, Decision Tree
  - **Regression**: Random Forest, XGBoost, Linear Regression, Ridge Regressor
- **Dual Execution**: Predicts both **Rain Occurrence & Probability** and **Rainfall Volume (mm)** on the same web interface in Google Colab!

In [ ]:
# Step 1: Install required dependencies
!pip install -q flask scikit-learn pandas numpy xgboost matplotlib seaborn

In [ ]:
# Step 2: Download dataset and perform Feature Engineering
import pandas as pd
import numpy as np

# Download Seattle weather dataset from official repository
url = "https://raw.githubusercontent.com/vega/vega-datasets/next/data/seattle-weather.csv"
print("Downloading stable Seattle rainfall dataset...")
df = pd.read_csv(url)
print(f"Dataset loaded successfully! Shape: {df.shape}")

# 1. Feature Engineering: Extract date parts
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

# 2. Define Features (X)
feature_cols = ["temp_max", "temp_min", "wind", "year", "month", "day"]
X = df[feature_cols]

# 3. Define Targets (y)
# Target A: Regression (predict continuous precipitation amount in mm)
y_regression = df["precipitation"]

# Target B: Classification (predict rain occurrence: 1 for rain/drizzle, 0 otherwise)
y_classification = df["weather"].apply(lambda x: 1 if x in ["rain", "drizzle"] else 0)

print("Features (X) Shape:", X.shape)
print("Regression Target (y_regression) Shape:", y_regression.shape)
print("Classification Target (y_classification) Shape:", y_classification.shape)
display(X.head())

In [ ]:
# Step 3: Split dataset for Regression and Classification
from sklearn.model_selection import train_test_split

# 1. Regression Split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_regression, test_size=0.2, random_state=42
)

# 2. Classification Split (Stratified to maintain class ratio)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_classification, test_size=0.2, random_state=42, stratify=y_classification
)

print("--- Regression Split ---")
print(f"Train Features: {X_train_reg.shape}, Test Features: {X_test_reg.shape}")

print("\n--- Classification Split ---")
print(f"Train Features: {X_train_clf.shape}, Test Features: {X_test_clf.shape}")

In [ ]:
# Step 4: Train multiple Classification and Regression models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, mean_absolute_error, mean_squared_error, r2_score

# 1. Train Classification Models
clf_models = {
    "random_forest": RandomForestClassifier(random_state=42, n_estimators=100),
    "xgboost": XGBClassifier(random_state=42, n_estimators=100, eval_metric="logloss"),
    "logistic_regression": LogisticRegression(random_state=42, max_iter=1000),
    "decision_tree": DecisionTreeClassifier(random_state=42)
}

print("=== Training Classification Models ===")
clf_metrics = []
for name, model in clf_models.items():
    model.fit(X_train_clf, y_train_clf)
    y_pred = model.predict(X_test_clf)
    acc = accuracy_score(y_test_clf, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test_clf, y_pred, average="binary")
    clf_metrics.append({
        "Model": name.replace("_", " ").title(),
        "Accuracy": f"{acc * 100:.2f}%",
        "Precision": f"{prec * 100:.2f}%",
        "Recall": f"{rec * 100:.2f}%",
        "F1-Score": f"{f1 * 100:.2f}%"
    })

clf_metrics_df = pd.DataFrame(clf_metrics)
display(clf_metrics_df)

# 2. Train Regression Models
reg_models = {
    "random_forest": RandomForestRegressor(random_state=42, n_estimators=100),
    "xgboost": XGBRegressor(random_state=42, n_estimators=100),
    "linear_regression": LinearRegression(),
    "ridge": Ridge(alpha=1.0, random_state=42)
}

print("\n=== Training Regression Models ===\n")
reg_metrics = []
for name, model in reg_models.items():
    model.fit(X_train_reg, y_train_reg)
    y_pred = np.clip(model.predict(X_test_reg), 0, None)
    mae = mean_absolute_error(y_test_reg, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    r2 = r2_score(y_test_reg, y_pred)
    reg_metrics.append({
        "Model": name.replace("_", " ").title(),
        "MAE (mm)": round(mae, 2),
        "RMSE (mm)": round(rmse, 2),
        "R2 Score": round(r2, 3)
    })

reg_metrics_df = pd.DataFrame(reg_metrics)
display(reg_metrics_df)

In [ ]:
# Step 5: Visualizations & Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix for Random Forest Classifier
rf_preds = clf_models["random_forest"].predict(X_test_clf)
cm = confusion_matrix(y_test_clf, rf_preds, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Rain", "Rain/Drizzle"], yticklabels=["No Rain", "Rain/Drizzle"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Random Forest Classifier - Confusion Matrix")

# 2. Regression MAE Comparison Barplot
sns.barplot(x="Model", y="MAE (mm)", data=reg_metrics_df, ax=axes[1], palette="Blues_d", hue="Model", legend=False)
axes[1].set_title("Regression Error (MAE mm - Lower is Better)")
axes[1].set_ylabel("Mean Absolute Error (mm)")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Create the HTML Template inside Colab
import os
os.makedirs("templates", exist_ok=True)

html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Seattle Weather &amp; Rainfall Intelligence Engine</title>
    <style>
        :root {
            --primary: #2563eb;
            --card-bg: #ffffff;
            --bg: #f1f5f9;
            --text-main: #0f172a;
            --text-muted: #64748b;
            --border: #e2e8f0;
        }
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            background-color: var(--bg);
            color: var(--text-main);
            padding: 25px 15px;
            display: flex;
            justify-content: center;
            min-height: 100vh;
        }
        .container {
            background-color: var(--card-bg);
            border-radius: 16px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.08);
            padding: 30px;
            max-width: 650px;
            width: 100%;
        }
        .header { text-align: center; margin-bottom: 22px; }
        .badge { display: inline-block; background: #e0e7ff; color: #4338ca; padding: 4px 12px; border-radius: 999px; font-size: 11px; font-weight: 700; text-transform: uppercase; margin-bottom: 8px; }
        h1 { color: #1e3a8a; font-size: 24px; font-weight: 800; }
        .subtitle { font-size: 13.5px; color: var(--text-muted); margin-top: 5px; }
        
        .preset-bar {
            background: #f8fafc;
            border: 1px solid var(--border);
            border-radius: 10px;
            padding: 10px 14px;
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            flex-wrap: wrap;
            gap: 8px;
        }
        .preset-label { font-size: 12px; font-weight: 700; color: #334155; }
        .preset-btn { background: #ffffff; border: 1px solid #cbd5e1; padding: 5px 10px; border-radius: 6px; font-size: 12px; cursor: pointer; font-weight: 600; }
        .preset-btn:hover { background: #e2e8f0; }

        .results-box {
            background: #ffffff;
            border: 2px solid #93c5fd;
            border-radius: 12px;
            padding: 18px;
            margin-bottom: 22px;
            box-shadow: 0 4px 12px rgba(37,99,235,0.08);
        }
        .results-box h2 { font-size: 16px; color: #1e3a8a; margin-bottom: 14px; border-bottom: 1px solid var(--border); padding-bottom: 6px; }
        .results-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 14px;
        }
        .result-card {
            padding: 14px;
            border-radius: 10px;
            border: 1px solid var(--border);
        }
        .card-rain { background: #fff1f2; border-color: #fecdd3; }
        .card-norain { background: #f0fdf4; border-color: #bbf7d0; }
        .card-reg { background: #f0f9ff; border-color: #bae6fd; }
        .tag { font-size: 10px; text-transform: uppercase; font-weight: 700; color: #64748b; }
        .status-val { font-size: 18px; font-weight: 800; margin: 4px 0; }
        .status-rain { color: #dc2626; }
        .status-norain { color: #16a34a; }
        .volume-val { font-size: 26px; font-weight: 800; color: #0284c7; margin: 4px 0; }
        .sub-info { font-size: 11.5px; color: #64748b; margin-top: 4px; }

        .form-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 12px;
        }
        .form-group { margin-bottom: 10px; }
        label { display: block; font-weight: 600; font-size: 12.5px; color: #475569; margin-bottom: 4px; }
        input[type="number"], select {
            width: 100%;
            padding: 9px 11px;
            border: 1.5px solid #cbd5e1;
            border-radius: 8px;
            font-size: 13.5px;
            background-color: #f8fafc;
        }
        input[type="number"]:focus, select:focus {
            outline: none;
            border-color: #2563eb;
            background: #fff;
        }
        .algo-grid {
            grid-column: span 2;
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 12px;
            background: #f8fafc;
            padding: 12px;
            border-radius: 8px;
            border: 1px solid var(--border);
            margin-top: 6px;
        }
        button.submit-btn {
            grid-column: span 2;
            padding: 13px;
            background: #2563eb;
            border: none;
            border-radius: 8px;
            color: white;
            font-size: 15px;
            font-weight: 700;
            cursor: pointer;
            margin-top: 14px;
            transition: background 0.2s;
        }
        button.submit-btn:hover { background: #1d4ed8; }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <span class="badge">Project 11 &bull; Dual-Model Architecture</span>
            <h1>Seattle Weather App ⛅</h1>
            <p class="subtitle">Simultaneous Classification &amp; Regression Inference</p>
        </div>

        <div class="preset-bar">
            <span class="preset-label">⚡ Presets:</span>
            <button type="button" class="preset-btn" onclick="setPreset(14.5, 6.2, 4.1, 2025, 11, 18)">🌧️ Rainy Fall</button>
            <button type="button" class="preset-btn" onclick="setPreset(27.8, 15.6, 2.3, 2025, 7, 20)">☀️ Sunny Summer</button>
            <button type="button" class="preset-btn" onclick="setPreset(7.2, 1.1, 3.5, 2025, 1, 15)">❄️ Chilly Winter</button>
        </div>

        {% if has_prediction %}
            <div class="results-box">
                <h2>🎯 Dual Predictive Analysis Outputs</h2>
                <div class="results-grid">
                    <div class="result-card {% if rain_status == 'Yes' %}card-rain{% else %}card-norain{% endif %}">
                        <div class="tag">Classification</div>
                        <div class="status-val {% if rain_status == 'Yes' %}status-rain{% else %}status-norain{% endif %}">
                            {% if rain_status == 'Yes' %}🌧️ Rain (Yes){% else %}☀️ No Rain{% endif %}
                        </div>
                        <div class="sub-info">Model: <strong>{{ selected_clf_name }}</strong></div>
                        <div class="sub-info">Confidence: <strong>{{ rain_prob }}%</strong></div>
                    </div>

                    <div class="result-card card-reg">
                        <div class="tag">Regression</div>
                        <div class="volume-val">💧 {{ "%.2f"|format(rain_amount) }} <span style="font-size:14px;">mm</span></div>
                        <div class="sub-info">Model: <strong>{{ selected_reg_name }}</strong></div>
                        <div class="sub-info">Category: <strong>{{ rain_category }}</strong></div>
                    </div>
                </div>
            </div>
        {% endif %}

        <form action="/predict" method="POST">
            <div class="form-grid">
                <div class="form-group">
                    <label for="temp_max">Max Temp (°C)</label>
                    <input type="number" step="0.1" id="temp_max" name="temp_max" value="{{ form_data.temp_max }}" required>
                </div>
                <div class="form-group">
                    <label for="temp_min">Min Temp (°C)</label>
                    <input type="number" step="0.1" id="temp_min" name="temp_min" value="{{ form_data.temp_min }}" required>
                </div>
                <div class="form-group">
                    <label for="wind">Wind Speed (m/s)</label>
                    <input type="number" step="0.1" id="wind" name="wind" value="{{ form_data.wind }}" required>
                </div>
                <div class="form-group">
                    <label for="year">Year</label>
                    <input type="number" id="year" name="year" value="{{ form_data.year }}" required>
                </div>
                <div class="form-group">
                    <label for="month">Month (1-12)</label>
                    <input type="number" id="month" name="month" min="1" max="12" value="{{ form_data.month }}" required>
                </div>
                <div class="form-group">
                    <label for="day">Day (1-31)</label>
                    <input type="number" id="day" name="day" min="1" max="31" value="{{ form_data.day }}" required>
                </div>

                <div class="algo-grid">
                    <div class="form-group">
                        <label for="clf_algo">Classification Algorithm</label>
                        <select id="clf_algo" name="clf_algo">
                            <option value="random_forest" {% if form_data.selected_clf == 'random_forest' %}selected{% endif %}>Random Forest Classifier</option>
                            <option value="xgboost" {% if form_data.selected_clf == 'xgboost' %}selected{% endif %}>XGBoost Classifier</option>
                            <option value="logistic_regression" {% if form_data.selected_clf == 'logistic_regression' %}selected{% endif %}>Logistic Regression</option>
                            <option value="decision_tree" {% if form_data.selected_clf == 'decision_tree' %}selected{% endif %}>Decision Tree Classifier</option>
                        </select>
                    </div>
                    <div class="form-group">
                        <label for="reg_algo">Regression Algorithm</label>
                        <select id="reg_algo" name="reg_algo">
                            <option value="random_forest" {% if form_data.selected_reg == 'random_forest' %}selected{% endif %}>Random Forest Regressor</option>
                            <option value="xgboost" {% if form_data.selected_reg == 'xgboost' %}selected{% endif %}>XGBoost Regressor</option>
                            <option value="linear_regression" {% if form_data.selected_reg == 'linear_regression' %}selected{% endif %}>Linear Regression</option>
                            <option value="ridge" {% if form_data.selected_reg == 'ridge' %}selected{% endif %}>Ridge Regressor</option>
                        </select>
                    </div>
                </div>

                <button type="submit" class="submit-btn">⚡ Analyze with Both Algorithms</button>
            </div>
        </form>
    </div>

    <script>
        function setPreset(tmax, tmin, w, y, m, d) {
            document.getElementById("temp_max").value = tmax;
            document.getElementById("temp_min").value = tmin;
            document.getElementById("wind").value = w;
            document.getElementById("year").value = y;
            document.getElementById("month").value = m;
            document.getElementById("day").value = d;
        }
    </script>
</body>
</html>"""

with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(html_content)
print("Template created successfully at templates/index.html!")

In [ ]:
# Step 7: Launch Flask Application inside Google Colab
import socket
import threading
import time
from flask import Flask, render_template, request
try:
    from google.colab import output
    in_colab = True
except ImportError:
    in_colab = False

app = Flask(__name__)

clf_names = {
    "random_forest": "Random Forest Classifier",
    "xgboost": "XGBoost Classifier",
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree Classifier"
}

reg_names = {
    "random_forest": "Random Forest Regressor",
    "xgboost": "XGBoost Regressor",
    "linear_regression": "Linear Regression",
    "ridge": "Ridge Regressor"
}

@app.route("/")
def home():
    default_form = {
        "temp_max": 14.5,
        "temp_min": 6.2,
        "wind": 4.1,
        "year": 2025,
        "month": 11,
        "day": 18,
        "selected_clf": "random_forest",
        "selected_reg": "random_forest"
    }
    return render_template("index.html", has_prediction=False, form_data=default_form)

@app.route("/predict", methods=["POST"])
def predict():
    temp_max = float(request.form["temp_max"])
    temp_min = float(request.form["temp_min"])
    wind = float(request.form["wind"])
    year = int(request.form["year"])
    month = int(request.form["month"])
    day = int(request.form["day"])

    selected_clf = request.form["clf_algo"]
    selected_reg = request.form["reg_algo"]

    form_data = {
        "temp_max": temp_max, "temp_min": temp_min, "wind": wind,
        "year": year, "month": month, "day": day,
        "selected_clf": selected_clf, "selected_reg": selected_reg
    }

    input_data = pd.DataFrame([[temp_max, temp_min, wind, year, month, day]], columns=feature_cols)

    # 1. Run Classification
    clf_model = clf_models[selected_clf]
    clf_pred = int(clf_model.predict(input_data)[0])
    rain_status = "Yes" if clf_pred == 1 else "No"

    # Probability estimation
    if hasattr(clf_model, "predict_proba"):
        rain_prob = round(float(clf_model.predict_proba(input_data)[0][1]) * 100, 1)
    else:
        rain_prob = 85.0 if clf_pred == 1 else 15.0

    # 2. Run Regression
    reg_model = reg_models[selected_reg]
    predicted_volume = float(reg_model.predict(input_data)[0])
    rain_amount = max(0.0, predicted_volume)
    if rain_status == "No" and rain_amount < 1.0:
        rain_amount = 0.0

    # Category
    if rain_status == "No" or rain_amount < 0.1:
        rain_category = "No Rain (Dry / Clear)"
    elif rain_amount <= 2.5:
        rain_category = "Light Rain / Drizzle"
    elif rain_amount <= 10.0:
        rain_category = "Moderate Rain"
    else:
        rain_category = "Heavy Rain"

    return render_template(
        "index.html",
        has_prediction=True,
        form_data=form_data,
        selected_clf_name=clf_names[selected_clf],
        selected_reg_name=reg_names[selected_reg],
        rain_status=rain_status,
        rain_prob=rain_prob,
        rain_amount=rain_amount,
        rain_category=rain_category
    )

def find_available_port(start=5000):
    p = start
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("localhost", p)) != 0:
                return p
            p += 1

free_port = find_available_port(5005)

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=free_port, use_reloader=False), daemon=True).start()
time.sleep(2.0)

print("=" * 60)
print(f"Multi-Algorithm App is LIVE on Port {free_port}!")
print("=" * 60)
if in_colab:
    output.serve_kernel_port_as_iframe(free_port)
    print("\nOr click the window link to open in a separate tab:")
    output.serve_kernel_port_as_window(free_port)
else:
    print(f"Open your browser at: http://localhost:{free_port}")

In [ ]:
# Step 8: Optional - Public Mobile Access using localtunnel
!npm install -g localtunnel

import urllib.request
public_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()
print("\n" + "=" * 60)
print(f"YOUR ENDPOINT PASSWORD (IP): {public_ip}")
print("=" * 60)
print("Instructions:")
print("1. Click the localtunnel URL below when generated.")
print("2. Paste the Endpoint Password (IP) above into the prompt to view the app.")
print("=" * 60 + "\n")

!lt --port {free_port}